[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/05-points-of-interest.ipynb)

# Points of Interest with SocialMapper

A **point of interest** (POI) is any specific location that someone might find useful or interesting -- a restaurant, a hospital, a school, a park, a bus stop. In spatial analysis, discovering *what is nearby* is one of the most fundamental questions. Where is the nearest grocery store? How many pharmacies can a resident reach within a 10-minute drive? Are there more coffee shops near affluent neighborhoods than near lower-income ones?

SocialMapper answers these questions by combining two powerful open-source systems: **OpenStreetMap** for POI data and **Valhalla** for real-world routing. The result is a POI search that goes far beyond simple radius queries -- it can tell you not just *what* is nearby, but *how long it actually takes to get there* by car, on foot, or by bicycle.

In this notebook you will learn how to:

1. **Search for POIs** near a location using `get_poi`
2. **Understand POI fields** and the data each result contains
3. **Browse the category taxonomy** -- 10 high-level groups covering hundreds of OSM tag types
4. **Filter by category** to find specific types of places
5. **Use travel-time-aware search** to find POIs within an actual driving or walking isochrone
6. **Visualize POI distributions** with pie charts, histograms, scatter plots, and bar charts
7. **Compare walking vs driving** reachability for the same time budget
8. **Overlay POIs on a demographic choropleth** combining census data with POI locations
9. **Import custom POIs from CSV** using `import_poi_csv`

We will use **Seattle, Washington** as our study area throughout.

## Setup

In [ ]:
# Uncomment to install on Google Colab:
# !pip install 'socialmapper @ git+https://github.com/mihiarc/socialmapper.git'

from socialmapper import (
    get_poi,
    import_poi_csv,
    create_isochrone,
    get_census_blocks,
    get_census_data,
    create_map,
)
from socialmapper.poi_categorization import POI_CATEGORY_MAPPING

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

## What Is OpenStreetMap?

Before we start querying POIs, it is worth understanding where the data comes from.

**OpenStreetMap (OSM)** is a collaborative, open-source map of the entire world. Think of it as Wikipedia for geographic data -- instead of encyclopedia articles, contributors map roads, buildings, parks, businesses, and millions of other features. Since its founding in 2004, OSM has grown to over **10 million registered contributors** and contains billions of geographic features spanning every continent.

For points of interest specifically, OSM is remarkably comprehensive. Contributors tag real-world locations with structured metadata:

- **Restaurants, cafes, and bars** with cuisine types, opening hours, and contact information
- **Hospitals, clinics, and pharmacies** with specialties and emergency services
- **Schools, universities, and libraries** with enrollment levels and public access
- **Parks, playgrounds, and sports facilities** with amenity details
- **Transit stops, parking lots, and fuel stations** with capacity and service information

SocialMapper queries OSM through the **Overpass API**, a powerful read-only interface that lets you search for features by type, location, and tags. When you call `get_poi()`, SocialMapper constructs an Overpass query behind the scenes, fetches the results, categorizes them into human-readable groups, and computes distances from your origin point.

Because OSM is community-maintained, coverage varies by region. Urban areas in North America, Europe, and East Asia tend to have excellent POI coverage. Rural areas may have fewer tagged features. This is an important consideration when interpreting your results.

## How POI Search Works

SocialMapper's `get_poi` function operates in two distinct modes depending on whether you provide a `travel_time` parameter. Understanding the difference is critical because it affects what distances mean, how results are sorted, and how many POIs you will find.

### Mode 1: Radius Search (no `travel_time`)

When you call `get_poi("Seattle, WA")` without a `travel_time`, the function:

1. Geocodes "Seattle, WA" to coordinates (47.6062, -122.3321)
2. Creates a **5 km radius circle** around that point
3. Queries OpenStreetMap for all POIs inside that circle
4. Computes the **geodesic (straight-line) distance** from the origin to each POI
5. Sorts results by `distance_km` (nearest first)

This mode is fast and simple, but the distances are "as the crow flies" -- they do not account for roads, bridges, one-way streets, or other routing realities.

### Mode 2: Travel-Time Search (with `travel_time`)

When you add `travel_time=10`, the function:

1. Geocodes the location to coordinates
2. Creates an **isochrone** -- the actual area reachable within 10 minutes by the specified travel mode
3. Queries OpenStreetMap for POIs inside that isochrone polygon
4. Computes **actual routed distances and travel times** via the Valhalla matrix API
5. Sorts results by `travel_time_minutes` (fastest to reach first)

This mode is slower (it makes additional API calls) but far more realistic. It answers the question people actually care about: "How long will it take me to *get there*?"

### Why Routed Distance Matters

Consider a coffee shop that is 0.5 km away in a straight line -- but on the other side of a highway with no pedestrian crossing. By foot, reaching it might require a 2 km detour to the nearest overpass, turning a 6-minute walk into a 25-minute trek. The straight-line distance is misleading; the routed travel time tells the real story.

This distinction is especially important for equity analysis. A neighborhood might appear well-served by nearby amenities on a radius map, but if those amenities are separated by physical barriers (rivers, highways, rail lines), actual access could be far worse than the straight-line distance suggests.

## The POI Category Taxonomy

OpenStreetMap uses a free-form tagging system where any contributor can assign any key-value pair to a feature. This flexibility is powerful but can be chaotic -- a restaurant might be tagged `amenity=restaurant`, `cuisine=italian`, `name=Bella Napoli`, and dozens of other attributes.

SocialMapper organizes this chaos into **10 high-level categories**, each encompassing dozens of specific OSM tag values. This lets you search for broad concepts like "healthcare" without needing to know every possible OSM tag for medical facilities.

Here are the categories with example sub-types:

In [ ]:
# Display each category with its tag count and representative examples
example_subtypes = {
    "food_and_drink": ["restaurant", "cafe", "bar", "bakery", "fast_food"],
    "shopping": ["supermarket", "clothes", "electronics", "convenience", "books"],
    "education": ["school", "university", "library", "kindergarten", "college"],
    "healthcare": ["hospital", "clinic", "pharmacy", "dentist", "veterinary"],
    "transportation": ["bus_station", "parking", "fuel", "bicycle_rental", "taxi"],
    "recreation": ["park", "cinema", "museum", "sports_centre", "playground"],
    "services": ["bank", "post_office", "police", "lawyer", "laundry"],
    "accommodation": ["hotel", "hostel", "guest_house", "camp_site", "motel"],
    "religious": ["place_of_worship", "church", "mosque", "temple", "synagogue"],
    "utilities": ["toilets", "drinking_water", "recycling", "vending_machine", "post_box"],
}

print(f"{'Category':<20} {'Tags':>5}   Example sub-types")
print("-" * 80)
for category, tags in POI_CATEGORY_MAPPING.items():
    examples = ", ".join(example_subtypes.get(category, tags[:5]))
    print(f"{category:<20} {len(tags):>5}   {examples}")

total_tags = sum(len(tags) for tags in POI_CATEGORY_MAPPING.values())
print(f"\nTotal: {len(POI_CATEGORY_MAPPING)} categories covering {total_tags} OSM tag values")

Each category casts a wide net. When you search for `categories=["healthcare"]`, SocialMapper looks for hospitals, clinics, pharmacies, dentists, veterinarians, nursing homes, physiotherapists, and many more. You do not need to enumerate each sub-type yourself.

## Basic POI Search

Let us start with the simplest possible query: find POIs near downtown Seattle with no category filter and no travel-time constraint. This uses the default 5 km radius and returns results sorted by straight-line distance.

In [ ]:
pois = get_poi("Seattle, WA", limit=50)

print(f"POIs found: {len(pois)}")
print(f"\n{'Name':<40} {'Category':<20} {'Distance':>10}")
print("-" * 72)
for poi in pois[:10]:
    print(f"{poi['name'][:38]:<40} {poi['category']:<20} {poi['distance_km']:>8.2f} km")

## Inspecting POI Fields

Each POI is returned as a Python dictionary with a consistent set of fields. Let us examine one in detail to understand what data is available.

In [ ]:
sample_poi = pois[0]

print("POI Fields:")
print("=" * 60)
for key, value in sample_poi.items():
    if key == "tags":
        print(f"  {key + ':':<22} <dict with {len(value)} OSM tags>")
    else:
        print(f"  {key + ':':<22} {value}")

print("\nOSM Tags (first 5):")
for i, (tag_key, tag_value) in enumerate(sample_poi["tags"].items()):
    if i >= 5:
        print(f"  ... and {len(sample_poi['tags']) - 5} more")
        break
    print(f"  {tag_key}: {tag_value}")

**Field reference:**

| Field | Type | Description |
|---|---|---|
| `name` | str | The POI's name from OpenStreetMap |
| `category` | str | SocialMapper's high-level category (e.g., `food_and_drink`) |
| `lat` | float | Latitude in decimal degrees |
| `lon` | float | Longitude in decimal degrees |
| `distance_km` | float | Geodesic (straight-line) distance from origin |
| `address` | str or None | Street address if available in OSM |
| `tags` | dict | Raw OpenStreetMap tags (amenity, cuisine, opening_hours, etc.) |

When `travel_time` is provided, two additional fields appear:

| Field | Type | Description |
|---|---|---|
| `travel_time_minutes` | float | Actual routed travel time via Valhalla |
| `travel_distance_km` | float | Actual routed distance (follows roads) |

## Filtering by Category

In practice, you rarely want *all* POIs. You want specific types -- food near a new apartment, pharmacies near an aging population center, parks within walking distance of a school. The `categories` parameter accepts a list of one or more category names from the taxonomy above.

In [ ]:
# Search for food and drink establishments
food_pois = get_poi("Seattle, WA", categories=["food_and_drink"], limit=30)

print(f"Food & drink POIs found: {len(food_pois)}")
print(f"\n{'Name':<40} {'Distance':>10}")
print("-" * 52)
for poi in food_pois[:8]:
    print(f"{poi['name'][:38]:<40} {poi['distance_km']:>8.2f} km")

### Combining Multiple Categories

You can pass multiple categories to search for several types at once. This is useful for composite analyses -- for example, finding all *essential services* (healthcare + education) in an area.

In [ ]:
essential_pois = get_poi(
    "Seattle, WA",
    categories=["education", "healthcare"],
    limit=30,
)

print(f"Education + Healthcare POIs: {len(essential_pois)}")
print(f"\n{'Name':<40} {'Category':<15} {'Distance':>10}")
print("-" * 67)
for poi in essential_pois[:10]:
    print(f"{poi['name'][:38]:<40} {poi['category']:<15} {poi['distance_km']:>8.2f} km")

## Visualizing POI Categories

When you search without category filters, the results span many different types of places. A **pie chart** is an effective way to see the composition at a glance -- which categories dominate the area around your search point?

In [ ]:
# Fetch a broad set of POIs for visualization
all_pois = get_poi("Seattle, WA", limit=100)

# Count POIs by category
df_all = pd.DataFrame(all_pois)
category_counts = df_all["category"].value_counts()

# Create pie chart
fig, ax = plt.subplots(figsize=(8, 8))
colors = plt.cm.Set2.colors[:len(category_counts)]
wedges, texts, autotexts = ax.pie(
    category_counts.values,
    labels=category_counts.index,
    autopct="%1.0f%%",
    colors=colors,
    startangle=90,
    pctdistance=0.82,
)
for text in autotexts:
    text.set_fontsize(9)
ax.set_title(
    f"POI Category Distribution near Seattle, WA\n({len(all_pois)} POIs within 5 km radius)",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
plt.tight_layout()
plt.show()

## Distance Distribution

A **histogram** of `distance_km` values reveals how POIs are distributed spatially. Are most POIs clustered near the center, or spread evenly across the search area? This pattern varies significantly between dense urban cores and suburban environments.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

distances = df_all["distance_km"]
ax.hist(distances, bins=20, color="#5B9BD5", edgecolor="white", linewidth=0.8)
ax.axvline(
    distances.median(),
    color="#d63384",
    linestyle="--",
    linewidth=2,
    label=f"Median: {distances.median():.2f} km",
)
ax.set_xlabel("Distance from Origin (km)", fontsize=11)
ax.set_ylabel("Number of POIs", fontsize=11)
ax.set_title(
    "Distribution of POI Distances from Downtown Seattle",
    fontsize=13,
    fontweight="bold",
)
ax.legend(fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Distance statistics:")
print(f"  Minimum:  {distances.min():.2f} km")
print(f"  Median:   {distances.median():.2f} km")
print(f"  Mean:     {distances.mean():.2f} km")
print(f"  Maximum:  {distances.max():.2f} km")

## Scatter Plot: POIs on a Map

Plotting POI coordinates as a scatter plot with **category-based coloring** gives a quick spatial overview. While this is not a true geographic map with basemap tiles, it reveals clustering patterns and the spatial relationship between different types of amenities.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

# Assign a color to each category
unique_categories = df_all["category"].unique()
color_map = {cat: plt.cm.Set2(i / max(len(unique_categories) - 1, 1)) for i, cat in enumerate(unique_categories)}

for category in unique_categories:
    subset = df_all[df_all["category"] == category]
    ax.scatter(
        subset["lon"],
        subset["lat"],
        c=[color_map[category]],
        label=f"{category} ({len(subset)})",
        s=40,
        alpha=0.75,
        edgecolors="white",
        linewidths=0.5,
    )

# Mark the origin
ax.scatter(
    -122.3321, 47.6062,
    c="red", marker="*", s=200, zorder=5,
    label="Origin (Seattle center)",
    edgecolors="darkred", linewidths=0.8,
)

ax.set_xlabel("Longitude", fontsize=11)
ax.set_ylabel("Latitude", fontsize=11)
ax.set_title(
    "POI Locations by Category near Seattle, WA",
    fontsize=13,
    fontweight="bold",
)
ax.legend(loc="upper left", fontsize=8, framealpha=0.9)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Travel-Time-Aware Search

Now let us use the more powerful search mode. By passing `travel_time=10` and `travel_mode="drive"`, SocialMapper will:

1. Build a 10-minute driving isochrone around Seattle's center
2. Query OSM for shopping POIs inside that real-world boundary
3. Call the Valhalla matrix API to compute actual driving time and distance to each POI
4. Sort results by `travel_time_minutes`

The results now include two extra fields: `travel_time_minutes` and `travel_distance_km`.

In [ ]:
timed_pois = get_poi(
    "Seattle, WA",
    categories=["shopping"],
    travel_time=10,
    travel_mode="drive",
    limit=25,
)

print(f"Shopping POIs within a 10-minute drive: {len(timed_pois)}")
print(f"\n{'Name':<35} {'Straight-line':>13} {'Routed dist':>12} {'Travel time':>12}")
print("-" * 75)
for poi in timed_pois[:10]:
    straight = f"{poi['distance_km']:.2f} km"
    routed = f"{poi.get('travel_distance_km', 'N/A')} km"
    travel = f"{poi.get('travel_time_minutes', 'N/A')} min"
    print(f"{poi['name'][:33]:<35} {straight:>13} {routed:>12} {travel:>12}")

## Travel Time vs Straight-Line Distance

The scatter plot below compares `distance_km` (straight-line) on the x-axis with `travel_time_minutes` (actual routed) on the y-axis. If every road were a perfectly straight line, the points would form a tight line. In reality, road networks create detours, and the scatter reveals how much the actual travel time deviates from what straight-line distance would predict.

Points **above** the trend indicate places that are harder to reach than their straight-line distance suggests (perhaps across a bridge or around a body of water). Points **below** the trend are more accessible than expected (perhaps on a fast arterial road).

In [ ]:
# Filter to POIs that have valid travel time data
df_timed = pd.DataFrame(timed_pois)
df_valid = df_timed.dropna(subset=["travel_time_minutes", "travel_distance_km"])

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    df_valid["distance_km"],
    df_valid["travel_time_minutes"],
    c="#5B9BD5",
    s=50,
    alpha=0.7,
    edgecolors="white",
    linewidths=0.5,
)

# Add a diagonal reference line
max_dist = df_valid["distance_km"].max()
# Rough reference: ~2 min per km driving in urban area
ax.plot(
    [0, max_dist], [0, max_dist * 2],
    color="#d63384", linestyle="--", linewidth=1.5,
    label="Reference: 2 min/km",
    alpha=0.7,
)

ax.set_xlabel("Straight-Line Distance (km)", fontsize=11)
ax.set_ylabel("Actual Travel Time (minutes)", fontsize=11)
ax.set_title(
    "Travel Time vs Straight-Line Distance\nShopping POIs near Seattle (10-min drive)",
    fontsize=13,
    fontweight="bold",
)
ax.legend(fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## Walking vs Driving: How Travel Mode Changes Access

The same time budget yields dramatically different results depending on how you travel. A 15-minute drive in Seattle might cover 8-10 km, while a 15-minute walk covers roughly 1-1.2 km. This difference has profound implications for equity analysis: populations that depend on walking (elderly residents, children, people without car access) experience a fundamentally different geography of access than those who drive.

Let us compare the number of food and drink POIs reachable within 15 minutes by each mode.

In [ ]:
mode_counts = {}
for mode in ["walk", "drive"]:
    result = get_poi(
        "Seattle, WA",
        categories=["food_and_drink"],
        travel_time=15,
        travel_mode=mode,
        limit=100,
    )
    mode_counts[mode] = len(result)
    print(f"{mode:>5}: {len(result)} food & drink POIs within 15 minutes")

In [ ]:
# Bar chart comparing walking vs driving
fig, ax = plt.subplots(figsize=(7, 5))

modes = list(mode_counts.keys())
counts = list(mode_counts.values())
bar_colors = ["#66C2A5", "#5B9BD5"]

bars = ax.bar(modes, counts, color=bar_colors, width=0.5, edgecolor="white", linewidth=1.5)

# Add count labels on top of each bar
for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        str(count),
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold",
    )

ax.set_ylabel("Number of POIs", fontsize=11)
ax.set_title(
    "Food & Drink POIs Reachable Within 15 Minutes\nWalking vs Driving from Seattle Center",
    fontsize=13,
    fontweight="bold",
)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylim(0, max(counts) * 1.2)
plt.tight_layout()
plt.show()

if mode_counts["drive"] > 0:
    ratio = mode_counts["drive"] / max(mode_counts["walk"], 1)
    print(f"\nDriving provides access to {ratio:.1f}x more food & drink options "
          f"in the same 15-minute window.")

This disparity is one of the central findings in urban accessibility research. Neighborhoods designed around cars have very different service landscapes than those designed for pedestrians, and the gap is often largest for the most essential services.

## Mapping POIs on a Demographic Choropleth

The most powerful analysis in SocialMapper combines POI locations with demographic context. By overlaying POI markers on a choropleth of median income (or population, or age), you can visually explore questions like:

- Are food options concentrated in wealthier neighborhoods?
- Do lower-income areas have adequate healthcare access?
- Where are the service deserts?

This requires the full SocialMapper pipeline: isochrone, census block groups, census data, merge, and then `create_map` with `overlay_points`.

In [ ]:
# Step 1: Create a 15-minute driving isochrone around Seattle
iso = create_isochrone("Seattle, WA", travel_time=15, travel_mode="drive")
print(f"Isochrone area: {iso['properties']['area_sq_km']:.1f} sq km")

# Step 2: Get census block groups within the isochrone
blocks = get_census_blocks(polygon=iso)
print(f"Block groups:   {len(blocks)}")

# Step 3: Fetch median income data
census = get_census_data(iso, variables=["population", "median_income"])
print(f"Census records:  {len(census.data)}")

# Step 4: Merge census data onto block group geometries
merged_blocks = []
for block in blocks:
    geoid = block["geoid"]
    if geoid in census.data:
        merged_blocks.append({**block, **census.data[geoid]})

print(f"Merged blocks:   {len(merged_blocks)}")

In [ ]:
# Get food & drink POIs to overlay on the map
overlay_food_pois = get_poi("Seattle, WA", categories=["food_and_drink"], limit=30)

# Format as overlay points for create_map
overlay_points = [
    {"lat": p["lat"], "lon": p["lon"], "name": p["name"]}
    for p in overlay_food_pois[:20]
]

# Create the choropleth with POI overlay
poi_map = create_map(
    data=merged_blocks,
    column="median_income",
    title="Median Income with Food & Drink POIs -- Seattle, WA",
    overlay_boundary=iso,
    overlay_points=overlay_points,
    show_stats=True,
)

display(Image(data=poi_map.image_data))

The choropleth shading shows median household income by census block group, while the magenta dots mark food and drink locations. Look for patterns: do the POI dots cluster in high-income or low-income areas? Are there visible gaps? This type of combined visualization is a starting point for equity analysis and community planning.

## Importing POIs from CSV

Not all POI data comes from OpenStreetMap. You may have your own list of locations -- branch offices, client sites, competitor locations, proposed development sites -- stored in a CSV file. SocialMapper's `import_poi_csv` function reads a CSV and converts it into the same list-of-dicts format that `get_poi` returns, making it easy to integrate custom data into your analysis pipeline.

The function expects columns for name, latitude, longitude, and type. You can customize the column names with parameters if your CSV uses different headers.

**Parameters:**

```python
import_poi_csv(
    csv_path,                  # Path to the CSV file
    name_field="name",         # Column with POI names
    lat_field="latitude",      # Column with latitudes
    lon_field="longitude",     # Column with longitudes
    type_field="type",         # Column with POI type/category
)
```

Let us create a small CSV in a temporary file and import it.

In [ ]:
import tempfile
import os

# Create a sample CSV with Seattle landmarks
csv_content = """name,latitude,longitude,type
Pike Place Market,47.6097,-122.3425,market
Space Needle,47.6205,-122.3493,landmark
University of Washington,47.6553,-122.3035,university
Seattle Art Museum,47.6073,-122.3381,museum
Gas Works Park,47.6456,-122.3344,park
"""

with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False) as f:
    f.write(csv_content)
    csv_path = f.name

try:
    imported = import_poi_csv(csv_path)
    print(f"Imported {len(imported)} POIs from CSV:\n")
    for poi in imported:
        print(f"  {poi['name']:<30} ({poi['lat']:.4f}, {poi['lon']:.4f})  type={poi.get('category', poi.get('type', 'N/A'))}")
finally:
    os.unlink(csv_path)

The imported POIs are now in the same dictionary format as results from `get_poi`, so you can use them anywhere SocialMapper expects POI data -- as overlay points on maps, as input to further distance calculations, or as part of a comparison analysis.

## Summary

This notebook covered the full workflow for discovering, filtering, analyzing, and visualizing points of interest with SocialMapper. Here is a concise reference of the key concepts and API calls:

### API Quick Reference

| Task | Code |
|---|---|
| Search POIs by radius (5 km) | `get_poi("Seattle, WA")` |
| Filter by category | `get_poi(location, categories=["food_and_drink", "healthcare"])` |
| Travel-time search | `get_poi(location, travel_time=10, travel_mode="drive")` |
| Limit result count | `get_poi(location, limit=50)` |
| Import from CSV | `import_poi_csv("path/to/file.csv")` |
| List all categories | `from socialmapper.poi_categorization import POI_CATEGORY_MAPPING` |
| Overlay POIs on a map | `create_map(data, column, overlay_points=[{"lat": ..., "lon": ..., "name": ...}])` |

### Key Concepts

- **OpenStreetMap** provides the POI data -- over 10 million contributors maintaining a global map
- **Radius mode** (no `travel_time`) uses a 5 km circle and straight-line distances
- **Travel-time mode** uses an isochrone and actual routed distances via Valhalla
- **Routed distances** are more realistic than straight-line distances because they follow actual road networks
- **Category taxonomy** organizes hundreds of OSM tags into 10 searchable groups
- **Walking vs driving** comparisons reveal fundamental differences in accessibility
- **Choropleth overlays** combine POI locations with demographic context for equity analysis
- **CSV import** lets you integrate your own location data into the SocialMapper pipeline

**Next notebook:** [06 -- Multi-Location Comparison](06-multi-location-comparison.ipynb)